# Notebook 1: Code Generator with Tools

**Cursor Feature We're Building:** Chat — generate code from natural language, with file operations

In this notebook we build the core of any AI coding editor: an agent that takes a natural language prompt, generates code, and can read/write files. This is the foundation that Cursor's Chat and Agent Mode are built on.

### What's New

| Concept | Why |
|---|---|
| `@tool` decorator | Cleaner than `Tool()` class — schema auto-generated from type hints |
| `bind_tools` | Attach tools directly to the LLM so it decides when to call them |
| `ToolNode` | LangGraph's built-in node that executes tool calls automatically |
| `MessagesState` | Pre-built state with message list + reducer — replaces manual `TypedDict` |
| `astream_events` | Stream tokens in real-time as the agent thinks |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
print("API Key loaded" if api_key else "API Key NOT found")

## Setting up the LLM

We use `ChatOpenAI` pointed at OpenRouter. This is the same model interface you saw in Day 10, but now we'll attach tools to it.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
)

response = llm.invoke("Say hello in one sentence.")
print(response.content)

## Defining Tools with `@tool`

In Day 10 you created tools using `Tool(name=..., func=..., description=...)`. The `@tool` decorator is the modern approach — it reads the function's docstring and type hints to auto-generate the tool schema.

Every coding editor needs file operations. Let's build three tools an agent needs to work with code files.

In [ ]:
from langchain_core.tools import tool


@tool
def read_file(filepath: str) -> str:
    """Read the contents of a file and return it as a string."""
    with open(filepath, "r") as f:
        return f.read()


@tool
def write_file(filepath: str, content: str) -> str:
    """Write content to a file. Creates the file if it doesn't exist."""
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    with open(filepath, "w") as f:
        f.write(content)
    return f"File written: {filepath}"


@tool
def list_directory(directory: str) -> str:
    """List all files and directories in the given path."""
    entries = os.listdir(directory)
    return "\n".join(entries)


tools = [read_file, write_file, list_directory]

# The decorator auto-generates the schema from the docstring + type hints
for t in tools:
    print(f"{t.name}: {t.description}")
    print(f"  Schema: {t.args_schema.model_json_schema()}\n")

## `bind_tools` — Let the LLM Decide When to Use Tools

In Day 10 you used `AgentExecutor` which handled tool routing for you. With LangGraph, we have more control.

`bind_tools` tells the LLM about available tools. The LLM then returns **tool call messages** when it wants to use one, instead of plain text.

In [ ]:
llm_with_tools = llm.bind_tools(tools)

# When we ask something that needs a tool, the LLM returns a tool_call
response = llm_with_tools.invoke("What files are in the current directory?")

print("Content:", response.content)
print("Tool calls:", response.tool_calls)

In [ ]:
# When we ask something that does NOT need a tool, it responds normally
response = llm_with_tools.invoke("What is Python?")

print("Content:", response.content[:200])
print("Tool calls:", response.tool_calls)

The LLM doesn't execute the tool — it just says *which* tool to call and *with what arguments*. We need something to actually run it.

## Building the Agent Graph

In Day 11 you built graphs with `TypedDict` state and manual tool execution. Now we use two upgrades:

- **`MessagesState`** — a pre-built state that holds a list of messages with a built-in reducer (appends new messages automatically)
- **`ToolNode`** — a pre-built node that looks at the last AI message, executes any tool calls, and returns the results as messages

This is the **Agent Loop** pattern: the agent thinks, optionally calls tools, observes results, and repeats until done.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode


def agent(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


def should_continue(state: MessagesState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END


graph = StateGraph(MessagesState)

graph.add_node("agent", agent)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue, ["tools", END])
graph.add_edge("tools", "agent")

app = graph.compile()
print("Graph compiled")

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

This is the same graph pattern Cursor uses internally — an LLM node connected to tools in a loop. The LLM decides when to call a tool and when it's done.

## Running the Agent

In [ ]:
from langchain_core.messages import HumanMessage

result = app.invoke(
    {"messages": [HumanMessage(content="List the files in the current directory")]}
)

# print(result)

# for msg in result["messages"]:
#     print(f"{msg.type}: {msg.content[:200]}")
#     if hasattr(msg, 'tool_calls') and msg.tool_calls:
#         print(f"  -> Tool calls: {msg.tool_calls}")
#     print()

## Code Generation Task

Now a real coding task — the agent should generate code and write it to a file.

In [ ]:
result = app.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Create a Python file called 'generated/calculator.py' with a Calculator class that has:
- add, subtract, multiply, divide methods
- A history list that tracks all operations
- A get_history method that returns the history

Write the file using the write_file tool.
"""
            )
        ]
    }
)

for msg in result["messages"]:
    print(f"[{msg.type}]")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Tool: {tc['name']}({list(tc['args'].keys())})")
    else:
        print(f"  {msg.content[:300]}")
    print()

In [ ]:
# Verify the file was created
print(open("generated/calculator.py").read())

## Adding a System Prompt

Right now the agent has no personality or constraints. In Cursor, the system prompt shapes the agent's behavior — what kind of code to write, what conventions to follow. This is equivalent to Cursor Rules (`.cursorrules`).

In [ ]:
from langchain_core.messages import SystemMessage

SYSTEM_PROMPT = """You are an expert Python developer assistant. When generating code:
- Use type hints on all functions
- Add concise docstrings
- Follow PEP 8 conventions
- Prefer modern Python (3.10+) features like match/case where appropriate

You have access to file tools. Always write generated code to files."""

result = app.invoke(
    {
        "messages": [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(
                content="""
Create a file 'generated/data_processor.py' with a DataProcessor class that:
- Takes a list of dictionaries in __init__
- Has filter_by(key, value) -> returns filtered list
- Has group_by(key) -> returns dict of grouped items
- Has summarize() -> returns count, keys present, sample row
"""
            ),
        ]
    }
)

for msg in result["messages"]:
    if msg.type == "ai" and not msg.tool_calls:
        print(msg.content)

In [ ]:
print(open("generated/data_processor.py").read())

## Streaming — Real-time Output

When you use Cursor, you see tokens appear in real-time as the AI generates. That's streaming. LangGraph supports this via `astream_events`.

In [ ]:
async def stream_agent(user_message: str):
    inputs = {
        "messages": [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=user_message),
        ]
    }

    async for event in app.astream_events(inputs, version="v2"):
        if event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"]
            if chunk.content:
                print(chunk.content, end="", flush=True)
        elif event["event"] == "on_tool_start":
            print(f"\n--- Calling tool: {event['name']} ---")
        elif event["event"] == "on_tool_end":
            print(f"--- Tool done ---\n")


await stream_agent("List files in the 'generated' directory and read calculator.py")

## Multi-Turn Conversations

A coding agent needs to remember context across turns — "now modify the function you just created". We do this by passing the full message history back into the graph.

In [ ]:
# Turn 1: Create a file
messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(
        content="Create 'generated/logger.py' with a SimpleLogger class that writes timestamped messages to a log file."
    ),
]

result = app.invoke({"messages": messages})
messages = result["messages"]

print("=== Turn 1 complete ===")
print(open("generated/logger.py").read()[:300])

In [ ]:
# Turn 2: Modify it — the agent has full context from turn 1
messages.append(
    HumanMessage(
        content="""
Now read the logger.py file and add these features:
- Log levels: INFO, WARNING, ERROR
- A method to filter logs by level
Write the updated file.
"""
    )
)

result = app.invoke({"messages": messages})

print("=== Turn 2 complete ===")
print(open("generated/logger.py").read())

## Step-by-step Visibility

LangGraph lets you observe each step the agent takes. This is useful for debugging and for building UIs that show progress — similar to how Cursor shows "Searching files...", "Writing code..." etc.

In [ ]:
result = app.invoke(
    {
        "messages": [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(
                content="Read generated/calculator.py, then create generated/test_calculator.py with pytest tests for all methods."
            ),
        ]
    }
)

for i, msg in enumerate(result["messages"]):
    if msg.type == "system":
        print(f"Step {i}: [system] (system prompt)")
    elif msg.type == "human":
        print(f"Step {i}: [human] {msg.content[:80]}")
    elif msg.type == "ai":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                args_preview = {k: str(v)[:50] for k, v in tc["args"].items()}
                print(f"Step {i}: [agent] calls {tc['name']}({args_preview})")
        else:
            print(f"Step {i}: [agent] {msg.content[:100]}")
    elif msg.type == "tool":
        print(f"Step {i}: [tool:{msg.name}] returned {str(msg.content)[:80]}")
    print()

In [ ]:
print(open("generated/test_calculator.py").read())

## Recap

| What we built | Cursor equivalent |
|---|---|
| `@tool` with auto-schema | Cursor's internal tools (file read, write, search) |
| `bind_tools` + `ToolNode` loop | The core agent loop that powers Chat and Agent Mode |
| System prompt shaping behavior | Cursor Rules (`.cursorrules`) |
| Streaming with `astream_events` | Real-time token output in the Chat panel |
| Multi-turn message history | Conversation context in Chat |
| Step-by-step visibility | "Searching files...", "Writing code..." progress indicators |

### Design Patterns Used

- **Agent Loop** — Goal intake -> tool use -> observe result -> repeat
- **Tool Use** — Each tool has a clear description, strict schema, validated I/O

### Next Notebook

The agent can generate code, but has no idea if it works. In Notebook 2 we add **self-correction** — the agent executes its own code, catches errors, and fixes them automatically (like Cursor's Bugbot).

In [ ]:
# Cleanup generated files
import shutil

shutil.rmtree("generated", ignore_errors=True)
print("Cleaned up generated files.")